# DSIP Lab: train a medical image model on Colab / Kaggle (free GPU)

1. **Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Settings → Accelerator → **GPU**, and turn Internet **on**.
2. Set `CONFIG` below to your project's config, then run the cells top to bottom.

Educational use only, not for clinical diagnosis.

In [ ]:
REPO_URL = "https://github.com/tormgibbs/helmax.git"   # change if you forked the repo
CONFIG = "configs/malaria_cells.yaml"                   # your project's config

import os, subprocess
if not os.path.exists("helmax"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("helmax/model")
%pip install -q -e .
!nvidia-smi -L

## Get the dataset

**Option A (Kaggle notebook):** click *Add Input*, attach the dataset, then point `SOURCE` at it, e.g. `/kaggle/input/cell-images-for-detecting-malaria/cell_images`.

**Option B (Colab):** upload your `kaggle.json` API token and download with the Kaggle CLI (command below), or mount Google Drive.

In [ ]:
SOURCE = None  # e.g. "/kaggle/input/cell-images-for-detecting-malaria/cell_images"; None = use data.source from the config

# Colab + Kaggle CLI example (uncomment):
# from google.colab import files; files.upload()   # pick kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d iarunava/cell-images-for-detecting-malaria -p data/raw --unzip

src = f'--source "{SOURCE}"' if SOURCE else ""
!ls data/raw 2>/dev/null || echo "data/raw is empty - set SOURCE or download first"

## 1. Preview the preprocessing pipeline (figure for your report)

In [ ]:
!dsip preview -c {CONFIG} {src} --spectrum -n 4
from IPython.display import Image, display
import glob
display(Image(sorted(glob.glob("figures/*_preprocess.png"), key=os.path.getmtime)[-1]))

## 2. Prepare: split by class/patient, de-duplicate, balance, preprocess

In [ ]:
!dsip prepare -c {CONFIG} {src} --overwrite

## 3. Train (evaluates on the test set at the end)

In [ ]:
!dsip train -c {CONFIG} --device 0

## 4. Results

In [ ]:
import json, glob
run = sorted(glob.glob("runs/*/*/weights/best.pt"), key=os.path.getmtime)[-1].rsplit("/weights", 1)[0]
print("Run:", run)
print(json.dumps(json.load(open(f"{run}/eval_test/metrics.json")), indent=2)[:3000])
for f in ["results.png", "eval_test/confusion_matrix_normalized.png", "eval_test/roc_curve.png", "eval_test/confusion_matrix.png"]:
    p = f"{run}/{f}"
    if os.path.exists(p):
        display(Image(p, width=600))

## 5. Save your run (weights + plots + metrics)

In [ ]:
import shutil
zip_path = shutil.make_archive(run.replace("/", "_"), "zip", run)
print("Saved", zip_path)
# Colab: from google.colab import files; files.download(zip_path)
# Kaggle: the zip appears under Output in /kaggle/working once you copy it there:
# shutil.copy(zip_path, "/kaggle/working/")

## 6. Optional: live demo with a public link

In [ ]:
%pip install -q gradio
!dsip demo -c {CONFIG} --share